# Insurance Cross Sell Targeting

Propensity modeling with top-k campaign targeting and business segmentation.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_prep import load_insurance_data, basic_clean, stratified_split
from src.features import build_preprocessor, prepare_model_inputs
from src.modeling import (
    run_lazypredict_discovery,
    select_top3_eligible_families,
    run_manual_engineering_track,
    run_flaml_track,
    run_pycaret_track,
)
from src.targeting import build_lift_gain_table, generate_batch_target_list
from src.evaluation import build_leaderboard

SEED = 42
PROJECT_NAME = 'insurance-cross-sell-targeting'
PROJECT_ROOT = Path('.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'insurance_cross_sell'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Business Problem and Success Criteria

Prioritize outreach budget toward customers with strongest cross-sell propensity.

Primary metric: PR-AUC
Secondary: ROC-AUC
Tertiary: Precision@K / recall policy behavior

## 2) Dataset Access and Data Dictionary

Dataset: Playground Series S4E7 insurance cross-sell.

In [2]:
df = load_insurance_data(RAW_DIR, sample_frac=0.08, random_state=SEED)
df.shape

(920384, 12)

## 3) Data Cleaning and Leakage Checks

- Remove duplicates
- Validate binary target
- Hold out realistic validation split

In [3]:
df = basic_clean(df, target_col='Response')
train_df, holdout_df = stratified_split(df, target_col='Response', random_state=SEED)
train_df['Response'].mean(), holdout_df['Response'].mean()

(0.12301254775521624, 0.12301373881582164)

## 4) Feature Engineering

- Numeric and categorical preprocessing
- Imputation and one-hot encoding for robust scoring

In [4]:
preprocessor = build_preprocessor(train_df.drop(columns=['Response']))
X_train, X_holdout, y_train, y_holdout = prepare_model_inputs(
    train_df=train_df,
    holdout_df=holdout_df,
    target_col='Response',
    preprocessor=preprocessor,
)
X_train.shape, X_holdout.shape

((736307, 15), (184077, 15))

## 5) Validation Strategy

- Stratified holdout
- Business-threshold and top-k evaluation
- Lift/gain analysis for campaign planning

In [5]:
pd.DataFrame({
    'split': ['train', 'holdout'],
    'rows': [len(train_df), len(holdout_df)],
    'response_rate': [train_df['Response'].mean(), holdout_df['Response'].mean()]
})

,split,rows,response_rate
0,train,736307,0.123013
1,holdout,184077,0.123014


## 6) Track 1: LazyPredict Discovery Lab

In [6]:
lazy_table = run_lazypredict_discovery(X_train, X_holdout, y_train, y_holdout)
lazy_table.head(15)

,Model,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
0,GaussianNB,0.699167,0.772063,0.798099,0.748666,0.012049
1,NearestCentroid,0.700833,0.761944,0.761944,0.749867,0.046049
2,BernoulliNB,0.712500,0.746486,0.804692,0.758776,0.012850
3,Perceptron,0.739167,0.675952,0.782977,0.775633,0.019505
4,QuadraticDiscriminantAnalysis,0.790000,0.627586,0.789774,0.805809,0.020238
5,ExtraTreeClassifier,0.830000,0.614537,0.614537,0.829044,0.017524
6,DecisionTreeClassifier,0.805833,0.586830,0.586830,0.809641,0.028081
7,LabelSpreading,0.825833,0.581690,0.787459,0.820327,0.480966
8,LabelPropagation,0.824167,0.580734,0.791345,0.819178,0.415496
9,XGBClassifier,0.857500,0.569399,0.820553,0.834732,0.933114


## 7) Selection of Top 3 Eligible Models

In [7]:
eligible_table, top3_families = select_top3_eligible_families(
    lazy_table=lazy_table,
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    y_holdout=y_holdout,
    random_state=SEED,
)
eligible_table, top3_families

(           lazy_model               family    pr_auc   roc_auc  \
 0      LGBMClassifier             lightgbm  0.432140  0.876070   
 1  CatBoostClassifier             catboost  0.426748  0.874669   
 2  LogisticRegression  logistic_regression  0.330035  0.840660   
 
    precision_at_20pct  threshold  policy_utility  train_time_sec  eligible  \
 0            0.388456       0.49        0.486134       22.633046      True   
 1            0.387098       0.72        0.700040       78.139882      True   
 2            0.338395       0.40        0.471262        1.974099      True   
 
   eligibility_note  
 0         eligible  
 1         eligible  
 2         eligible  ,
 ['logistic_regression', 'catboost', 'lightgbm'])

## 8) Track 2: Manual Engineering Lab

Includes ranking/top-k policy and threshold tuning.

In [8]:
manual_results, manual_models, manual_scores = run_manual_engineering_track(
    top3_families=top3_families,
    X_train=X_train, y_train=y_train,
    X_holdout=X_holdout, y_holdout=y_holdout,
    random_state=SEED,
)
manual_results[['model_name', 'pr_auc', 'roc_auc', 'precision_at_20pct', 'threshold']]

,model_name,pr_auc,roc_auc,precision_at_20pct,threshold
0,lightgbm,0.433621,0.876174,0.388863,0.61
1,catboost,0.424714,0.873939,0.385278,0.90
2,logistic_regression,0.330035,0.840660,0.338395,0.40


## 9) Track 3: FLAML Optimization Lab

In [9]:
flaml_result = run_flaml_track(
    X_train=X_train, y_train=y_train,
    X_holdout=X_holdout, y_holdout=y_holdout,
    time_budget=120, random_state=SEED
)
flaml_result

[flaml.automl.logger: 06-11 23:59:46] {2375} INFO - task = classification


[flaml.automl.logger: 06-11 23:59:46] {2386} INFO - Evaluation method: cv


[flaml.automl.logger: 06-11 23:59:46] {2489} INFO - Minimizing error metric: 1-ap


[flaml.automl.logger: 06-11 23:59:46] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'xgboost', 'rf', 'extra_tree', 'lrl1']


[flaml.automl.logger: 06-11 23:59:46] {2911} INFO - iteration 0, current learner lgbm


...

[flaml.automl.logger: 06-12 00:03:27] {3359} INFO - retrain xgboost for 21.0s


[flaml.automl.logger: 06-12 00:03:27] {3362} INFO - retrained model: XGBClassifier(base_score=None, booster=None, callbacks=[],
              colsample_bylevel=0.9919147257144936, colsample_bynode=None,
              colsample_bytree=0.9549891298248687, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, gamma=None,
              grow_policy='lossguide', importance_type=None,
              interaction_constraints=None, learning_rate=0.5500317092437325,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=0, max_leaves=28,
              min_child_weight=0.9030514379170209, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=16,
              n_jobs=-1, num_parallel_tree=None, random_state=None, ...)


[flaml.automl.logger: 06-12 00:03:27] {2636} INFO - fit succeeded


[flaml.automl.logger: 06-12 00:03:27] {2637} INFO - Time taken to find the best model: 118.28522253036499


{'model_name': 'xgboost',
 'library_source': 'flaml',
 'pr_auc': 0.4187922867212997,
 'roc_auc': 0.8719961058713142,
 'precision_at_20pct': 0.38264294445198965,
 'recall_at_operating': 0.0001766472354707649,
 'threshold': 0.73,
 'policy_utility': 0.7000529941706412,
 'train_time_sec': 221.2746410339896,
 'infer_latency_ms': nan,
 'p95_latency_ms': nan,
 'interpretability_note': 'FLAML budget-aware targeting challenger',
 'best_config': {'n_estimators': 16,
  'max_leaves': 28,
  'min_child_weight': 0.9030514379170209,
  'learning_rate': 0.5500317092437325,
  'subsample': 0.7837846133419992,
  'colsample_bylevel': 0.9919147257144936,
  'colsample_bytree': 0.9549891298248687,
  'reg_alpha': 0.0009765625,
  'reg_lambda': 0.4502108984066631},
 'best_loss': 0.6062435138324325,
 'scores': array([4.4799700e-02, 1.4504775e-01, 6.0302091e-01, ..., 2.4227239e-01,
        3.6527756e-01, 2.5482316e-04], dtype=float32)}

## 10) Track 4: PyCaret Experiment Lab

In [10]:
pycaret_result = run_pycaret_track(
    train_df=train_df,
    holdout_df=holdout_df,
    target_col='Response',
    session_id=SEED,
    model_output_path=ARTIFACT_DIR / 'pycaret_insurance_cross_sell',
)
pycaret_result

{'model_name': 'pycaret_failed',
 'library_source': 'pycaret',
 'pr_auc': nan,
 'roc_auc': nan,
 'precision_at_20pct': nan,
 'recall_at_operating': nan,
 'threshold': nan,
 'policy_utility': nan,
 'train_time_sec': nan,
 'infer_latency_ms': nan,
 'p95_latency_ms': nan,
 'interpretability_note': "PyCaret unavailable or failed: ('Pycaret only supports python 3.9, 3.10, 3.11. Your actual Python version: ', sys.version_info(major=3, minor=12, micro=10, releaselevel='final', serial=0), 'Please DOWNGRADE your Python version.')",
 'scores': array([], dtype=float64),
 'status': 'failed'}

## 11) Unified Leaderboard and Final Model Ranking

In [11]:
leaderboard = build_leaderboard(
    project_name=PROJECT_NAME,
    lazy_results=eligible_table,
    manual_results=manual_results,
    flaml_result=flaml_result,
    pycaret_result=pycaret_result,
)
leaderboard.head(10)

,project_name,task_type,library_source,model_name,cv_metric_mean,cv_metric_std,holdout_primary_metric,holdout_secondary_metric,holdout_tertiary_metric,calibration_metric,train_time_sec,infer_latency_ms,p95_latency_ms,model_size_mb,retrain_time_sec,interpretability_note,rank_score,final_rank
0,insurance-cross-sell-targeting,binary_classification_targeting,manual,lightgbm,NaN,NaN,0.433621,0.876174,0.388863,NaN,13992.268137,2.172053,3.292296,NaN,13992.268137,Manual calibrated model: lightgbm,0.535308,1
1,insurance-cross-sell-targeting,binary_classification_targeting,lazypredict,lightgbm,NaN,NaN,0.432140,0.876070,0.388456,NaN,22.633046,NaN,NaN,NaN,22.633046,eligible,0.534385,2
2,insurance-cross-sell-targeting,binary_classification_targeting,lazypredict,catboost,NaN,NaN,0.426748,0.874669,0.387098,NaN,78.139882,NaN,NaN,NaN,78.139882,eligible,0.530798,3
3,insurance-cross-sell-targeting,binary_classification_targeting,manual,catboost,NaN,NaN,0.424714,0.873939,0.385278,NaN,152.276400,1.528612,1.617049,NaN,152.276400,Manual calibrated model: catboost,0.529133,4
4,insurance-cross-sell-targeting,binary_classification_targeting,flaml,xgboost,NaN,NaN,0.418792,0.871996,0.382643,NaN,221.274641,NaN,NaN,NaN,221.274641,FLAML budget-aware targeting challenger,0.524863,5
5,insurance-cross-sell-targeting,binary_classification_targeting,lazypredict,logistic_regression,NaN,NaN,0.330035,0.840660,0.338395,NaN,1.974099,NaN,NaN,NaN,1.974099,eligible,0.459363,6
6,insurance-cross-sell-targeting,binary_classification_targeting,manual,logistic_regression,NaN,NaN,0.330035,0.840660,0.338395,NaN,1.979524,0.043439,0.052767,NaN,1.979524,Manual calibrated model: logistic_regression,0.459363,7
7,insurance-cross-sell-targeting,binary_classification_targeting,pycaret,pycaret_failed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PyCaret unavailable or failed: ('Pycaret only ...,0.000000,8


In [12]:
leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_insurance_cross_sell.csv', index=False)
leaderboard[['model_name', 'library_source', 'holdout_primary_metric', 'holdout_tertiary_metric', 'final_rank']].head(10)

,model_name,library_source,holdout_primary_metric,holdout_tertiary_metric,final_rank
0,lightgbm,manual,0.433621,0.388863,1
1,lightgbm,lazypredict,0.432140,0.388456,2
2,catboost,lazypredict,0.426748,0.387098,3
3,catboost,manual,0.424714,0.385278,4
4,xgboost,flaml,0.418792,0.382643,5
5,logistic_regression,lazypredict,0.330035,0.338395,6
6,logistic_regression,manual,0.330035,0.338395,7
7,pycaret_failed,pycaret,NaN,NaN,8


## 12) Business Recommendation

State winner rationale, safer runner-up scenario, and excluded tradeoff.

## 13) Inference / Deployment Path

Generate batch target list by campaign budget.

In [13]:
winner = leaderboard.sort_values('final_rank').iloc[0]
if winner['library_source'] == 'manual' and winner['model_name'] in manual_scores:
    scores = manual_scores[winner['model_name']]
elif winner['library_source'] == 'flaml':
    scores = flaml_result.get('scores', np.array([]))
else:
    scores = pycaret_result.get('scores', np.array([]))

if len(scores) != len(holdout_df):
    scores = np.resize(scores, len(holdout_df))

id_col = 'id' if 'id' in holdout_df.columns else holdout_df.columns[0]
target_list = generate_batch_target_list(holdout_df[[id_col]].copy(), scores, id_col=id_col, top_target_pct=0.2, hold_pct=0.3)
target_list.to_csv(ARTIFACT_DIR / 'target_list.csv', index=False)
target_list.head()

,id,score,campaign_priority
5936,9647061,0.631051,target
156608,4234329,0.615943,target
158204,7838496,0.610990,target
17198,8441746,0.609509,target
66100,5970277,0.606839,target


In [14]:
lift_gain = build_lift_gain_table(holdout_df['Response'].astype(int).values, scores, n_bins=10)
lift_gain.to_csv(ARTIFACT_DIR / 'lift_gain_table.csv', index=False)
lift_gain

,decile,rows,positives,cumulative_positives,capture_rate,lift_vs_random
0,1,18408,8386,8386,0.370341,3.703409
1,2,18408,5930,14316,0.632220,3.161102
2,3,18407,4252,18568,0.819996,2.733322
3,4,18408,2753,21321,0.941574,2.353935
4,5,18408,1157,22478,0.992669,1.985338
5,6,18407,154,22632,0.999470,1.665783
6,7,18408,8,22640,0.999823,1.428319
7,8,18407,3,22643,0.999956,1.249945
8,9,18408,1,22644,1.000000,1.111111
9,10,18408,0,22644,1.000000,1.000000


## 14) Monitoring / Drift / Retraining Plan

Track PR-AUC, precision@20%, capture rate in top deciles, and campaign uplift stability.

## 15) Limitations and Next Steps

- Add treatment uplift modeling
- Add contact policy constraints (frequency caps)
- Add fairness and segment-level calibration diagnostics